# DS-CP reproduction with the authors' code (Task 2, DSA1)
Code: https://github.com/zhexiaolin/CP (Lin et al., ICML 2026). Data: the authors' saved MMLU outputs (LLM-Uncertainty-Bench).
Model: deepseek-llm-67b-base, 90% coverage target. We rerun the first 32 subject pairs and compare the pairs that also
appear in the authors' published results file. Standard CP = "LAC", DS-CP = "LAC_W" in the authors' files.

In [8]:
import os, shutil, subprocess
from pathlib import Path

os.chdir("/kaggle/working")
if not Path("dscp-authors").exists():
    subprocess.run(["git", "clone", "https://github.com/zhexiaolin/CP", "dscp-authors"], check=True)
os.chdir("/kaggle/working/dscp-authors")

# The script APPENDS to coverage_<model>.csv, so keep an untouched copy of the authors' results first.
Path("authors_results").mkdir(exist_ok=True)
for f in Path(".").glob("coverage_*.csv"):
    if not (Path("authors_results") / f.name).exists():
        shutil.copy(f, Path("authors_results") / f.name)
print(sorted(p.name for p in Path("authors_results").glob("*.csv")))

['coverage_Llama-2-70b-hf.csv', 'coverage_Qwen-72B.csv', 'coverage_Yi-34B.csv', 'coverage_deepseek-llm-67b-base.csv']


In [9]:
import time
import pandas as pd

MODEL = "deepseek-llm-67b-base"
RESULTS = f"coverage_{MODEL}.csv"
TARGET_NEW_PAIRS = 32      # pairs to rerun; about 14 of them overlap with the authors' published file

# start each run from the authors' file, so reruns of this notebook give the same rows
shutil.copy(Path("authors_results") / RESULTS, RESULTS)
n_authors = len(pd.read_csv(RESULTS))

def new_pairs():
    try:
        return len(pd.read_csv(RESULTS)) - n_authors
    except Exception:          # file is being written at this moment
        return -1

# The script ignores --model and works through its own list of models; deepseek comes first.
log = open("/kaggle/working/dscp_run_log.txt", "w")
proc = subprocess.Popen(["python", "main_mmlu.py", f"--model={MODEL}"], stdout=log, stderr=subprocess.STDOUT)
start = time.time()
while proc.poll() is None:
    time.sleep(30)
    n = new_pairs()
    print(f"{(time.time() - start) / 60:5.1f} min: {n} new pairs", flush=True)
    model_changed = any("Running for model" in line and MODEL not in line
                        for line in open("/kaggle/working/dscp_run_log.txt"))
    if n >= TARGET_NEW_PAIRS or model_changed:
        proc.terminate()
        proc.wait()
        break
log.close()
print("stopped after", new_pairs(), "new pairs")

  0.5 min: 2 new pairs
  1.0 min: 4 new pairs
  1.5 min: 5 new pairs
  2.0 min: 7 new pairs
  2.5 min: 11 new pairs
  3.0 min: 15 new pairs
  3.5 min: 17 new pairs
  4.0 min: 18 new pairs
  4.5 min: 19 new pairs
  5.0 min: 20 new pairs
  5.5 min: 21 new pairs
  6.0 min: 23 new pairs
  6.5 min: 25 new pairs
  7.0 min: 27 new pairs
  7.5 min: 28 new pairs
  8.0 min: 30 new pairs
  8.5 min: 32 new pairs
stopped after 32 new pairs


In [10]:
full = pd.read_csv(RESULTS)
authors_file = pd.read_csv(Path("authors_results") / RESULTS)
print("rows now:", len(full), "| authors' rows:", len(authors_file))
print("first rows still identical to the authors' file:",
      full.head(len(authors_file)).astype(str).equals(authors_file.astype(str)))
print("rerun pairs that also appear in the authors' file:",
      full.iloc[len(authors_file):].merge(authors_file, on=["old", "new"]).shape[0])

rows now: 168 | authors' rows: 136
first rows still identical to the authors' file: True
rerun pairs that also appear in the authors' file: 14


In [11]:
import ast

COLS = ["coverage_LAC", "setsize_LAC", "coverage_LAC_W", "setsize_LAC_W"]
def parse(df):
    df = df.copy()
    for col in COLS:
        df[col] = df[col].apply(lambda s: list(ast.literal_eval(s).values())[0])  # "{'base_icl1': 0.96}" -> 0.96
    return df

authors = parse(full.iloc[:n_authors])                              # the authors' published rows
rerun = parse(full.iloc[n_authors:n_authors + TARGET_NEW_PAIRS])    # our rerun rows
both = rerun.merge(authors, on=["old", "new"], suffixes=("", "_authors"))
print(f"{len(rerun)} pairs rerun, {len(both)} of them also in the authors' results")

pairwise = both[["old", "new", "coverage_LAC", "coverage_LAC_authors", "coverage_LAC_W", "coverage_LAC_W_authors"]].round(3)
pairwise.to_csv("/kaggle/working/task2_dscp_pairwise_check.csv", index=False)
display(pairwise)

rows = []
for method, col in [("standard CP", "LAC"), ("DS-CP", "LAC_W")]:
    for source, sfx in [("our rerun", ""), ("authors", "_authors")]:
        c, s = both[f"coverage_{col}{sfx}"], both[f"setsize_{col}{sfx}"]
        rows.append({"model": MODEL, "method": method, "source": source, "subject_pairs": len(both),
                     "mean_coverage": c.mean(), "worst_coverage": c.min(),
                     "share_pairs_below_90": (c < 0.90).mean(), "mean_set_size": s.mean()})
    rows[-2]["max_diff_vs_authors"] = (both[f"coverage_{col}"] - both[f"coverage_{col}_authors"]).abs().max()
repro = pd.DataFrame(rows)
repro.to_csv("/kaggle/working/task2_dscp_authors_repro.csv", index=False)
repro.round(3)

32 pairs rerun, 14 of them also in the authors' results


,old,new,coverage_LAC,coverage_LAC_authors,coverage_LAC_W,coverage_LAC_W_authors
0,geography,law,0.715,0.715,0.814,0.810
1,geography,business,0.926,0.926,0.979,0.979
2,geography,culture,0.906,0.906,0.939,0.939
3,geography,chemistry,0.683,0.683,0.845,0.853
4,geography,politics,0.864,0.864,0.910,0.910
5,computer science,psychology,0.937,0.937,0.925,0.932
6,computer science,law,0.860,0.860,0.862,0.862
7,computer science,business,0.982,0.982,0.985,0.985
8,computer science,economics,0.912,0.912,0.920,0.913
9,computer science,culture,0.939,0.939,0.950,0.946


,model,method,source,subject_pairs,mean_coverage,worst_coverage,share_pairs_below_90,mean_set_size,max_diff_vs_authors
0,deepseek-llm-67b-base,standard CP,our rerun,14,0.881,0.683,0.429,1.599,0.000
1,deepseek-llm-67b-base,standard CP,authors,14,0.881,0.683,0.429,1.599,NaN
2,deepseek-llm-67b-base,DS-CP,our rerun,14,0.921,0.814,0.214,1.833,0.041
3,deepseek-llm-67b-base,DS-CP,authors,14,0.923,0.810,0.214,1.879,NaN


In [12]:
from IPython.display import FileLink, display
os.chdir("/kaggle/working")
display(FileLink("task2_dscp_authors_repro.csv"), FileLink("task2_dscp_pairwise_check.csv"), FileLink("dscp_run_log.txt"))

/kaggle/working/task2_dscp_authors_repro.csv

/kaggle/working/task2_dscp_pairwise_check.csv

/kaggle/working/dscp_run_log.txt